# Multi-Agent Cooperation Patterns

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/07-multi-agent-cooperation-patterns

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
BRAND = '#6366f1'; TEAL = '#2dd4bf'; ROSE = '#fb7185'; YELLOW = '#fbbf24'
np.random.seed(0)

## 1. Voting-based cooperation

In voting-based cooperation, $N$ independent agents each answer the same question, and a coordinator aggregates the answers using a voting rule. We simulate each agent as a function that is correct with probability `p` and returns a random wrong answer otherwise.

**Majority vote** returns the label that received strictly more than half the votes (or the plurality if no majority exists).

In [ ]:
LABELS = ['positive', 'negative', 'neutral']
TRUE_LABEL = 'positive'

def simulated_agent(accuracy: float, rng: np.random.Generator) -> str:
    """Return the correct label with probability `accuracy`, else a random wrong label."""
    if rng.random() < accuracy:
        return TRUE_LABEL
    wrong = [l for l in LABELS if l != TRUE_LABEL]
    return rng.choice(wrong)

def majority_vote(answers: list[str]) -> str:
    """Return the most common answer (ties broken by first occurrence)."""
    counts = Counter(answers)
    return counts.most_common(1)[0][0]

def voting_coordinator(n_agents: int, accuracy: float, rng: np.random.Generator) -> str:
    answers = [simulated_agent(accuracy, rng) for _ in range(n_agents)]
    result = majority_vote(answers)
    print(f'  Agents answered: {answers}')
    print(f'  Majority vote  : {result}')
    return result

rng = np.random.default_rng(42)
print('N=3 agents, accuracy=0.6:')
voting_coordinator(3, 0.6, rng)
print()
print('N=7 agents, accuracy=0.6:')
voting_coordinator(7, 0.6, rng)

## 2. Condorcet's Jury Theorem — empirical verification

The theorem states that if each voter independently has probability $p > 0.5$ of being correct, the probability of the majority being correct is:

$$P(\text{majority correct}) = \sum_{k=\lceil N/2 \rceil}^{N} \binom{N}{k} p^k (1-p)^{N-k}$$

and this approaches 1 as $N \to \infty$. We plot the analytic curve and overlay a Monte-Carlo simulation to verify it.

In [ ]:
from math import comb

def condorcet_prob(n: int, p: float) -> float:
    """Analytic probability that majority of n voters is correct."""
    majority = (n // 2) + 1
    return sum(comb(n, k) * (p ** k) * ((1 - p) ** (n - k)) for k in range(majority, n + 1))

def monte_carlo_majority(n: int, p: float, trials: int = 10_000) -> float:
    """Empirical P(majority correct) via simulation."""
    rng = np.random.default_rng(0)
    votes = rng.random((trials, n)) < p          # True = correct vote
    majority_correct = votes.sum(axis=1) > n / 2
    return majority_correct.mean()

ns = list(range(1, 31, 2))  # odd numbers 1..29
configs = [
    (0.55, ROSE,   'p = 0.55'),
    (0.60, YELLOW, 'p = 0.60'),
    (0.70, TEAL,   'p = 0.70'),
]

fig, ax = plt.subplots()
for p, color, label in configs:
    analytic = [condorcet_prob(n, p) for n in ns]
    empirical = [monte_carlo_majority(n, p) for n in ns]
    ax.plot(ns, analytic,  color=color, linewidth=2, label=f'{label} (analytic)')
    ax.scatter(ns, empirical, color=color, s=20, alpha=0.6)

ax.axhline(1.0, color='#475569', linestyle='--', linewidth=1)
ax.set_xlabel('Number of agents N')
ax.set_ylabel('P(majority correct)')
ax.set_title("Condorcet's Jury Theorem")
ax.legend(facecolor='#1a1d27', edgecolor='#334155')
plt.tight_layout()
plt.show()

print('Analytic P(majority correct) for p=0.7:')
for n in [1, 3, 5, 11, 21]:
    print(f'  N={n:>2}: {condorcet_prob(n, 0.7):.4f}')

The dots (Monte-Carlo) track the analytic lines closely — verifying the formula. Notice how even a modest individual accuracy of 0.55 converges toward 1 given enough independent agents; and how much faster p=0.70 gets there.

## 3. Role-based cooperation

In role-based cooperation, agents have specialised roles: **Planner**, **Executor**, **Critic**, and **Summariser**. Each role gets a distinct system prompt (represented here as a Python function) and communicates via structured message dicts.

We simulate a toy task: `"write a Python function to sort a list"`.

In [ ]:
# Each 'agent' is a pure function: dict -> dict
# In a real system these would be LLM calls with different system prompts.

def planner(goal: str) -> dict:
    """Decomposes the goal into an ordered task list."""
    tasks = [
        'implement the sorting function',
        'write at least two test cases',
        'review for edge cases (empty list, single element)',
    ]
    return {'role': 'plan', 'goal': goal, 'tasks': tasks}

def executor(plan: dict) -> dict:
    """Carries out the first uncompleted task and returns an artifact."""
    code = (
        'def sort_list(items):\n'
        '    return sorted(items)\n'
        '\n'
        'assert sort_list([3, 1, 2]) == [1, 2, 3]\n'
        'assert sort_list([]) == []'
    )
    return {'role': 'artifact', 'code': code, 'tasks_done': plan['tasks'][:2]}

def critic(artifact: dict) -> dict:
    """Reviews the artifact and returns structured feedback."""
    issues = ['Missing test for duplicate elements, e.g. [2, 2, 1]']
    return {
        'role': 'critique',
        'issues': issues,
        'severity': 'minor',
        'approved': True,   # approved despite minor issue
    }

def summariser(artifact: dict, critique: dict) -> dict:
    """Compresses results into a compact handoff context."""
    summary = (
        f'sort_list(items) — uses Python built-in sorted(), O(n log n). '
        f'Approved: {critique["approved"]}. '
        f'Open issues: {critique["issues"]}'
    )
    return {'role': 'summary', 'text': summary}

# Orchestrator wires everything together
def role_based_pipeline(goal: str) -> str:
    print(f'Goal: {goal}\n')
    plan      = planner(goal)
    print(f'[Planner] tasks: {plan["tasks"]}')
    artifact  = executor(plan)
    print(f'[Executor] produced code:\n{artifact["code"]}')
    critique  = critic(artifact)
    print(f'[Critic] approved={critique["approved"]}, issues={critique["issues"]}')
    result    = summariser(artifact, critique)
    print(f'[Summariser] {result["text"]}')
    return result['text']

final = role_based_pipeline('write a Python function to sort a list')

Each role gets exactly the information it needs — no more. The Critic never sees the Planner's task list; it only sees the artifact. The Summariser condenses everything into a compact context for whatever comes next. This is separation of concerns in action.

## 4. Debate-based cooperation

Two agents argue opposing positions over $K$ rounds. A mediator scores each round and produces a final synthesised answer. We simulate a debate on the question: *"Should the team adopt microservices for this project?"*

In [ ]:
# Scripted arguments — in a real system these would be LLM calls.
DEBATE_SCRIPTS = {
    'pro': [
        'Microservices allow independent deployment; each service can scale separately based on load.',
        'The team already has Docker expertise. Operational overhead is lower than it appears.',
        'Independent services mean a failure in one does not cascade to the whole system.',
    ],
    'con': [
        'The team is small (3 engineers). Microservices add significant operational overhead: service discovery, distributed tracing, inter-service auth.',
        'Docker expertise is not the same as Kubernetes expertise. Orchestration complexity is routinely underestimated.',
        'Network latency between services is a new failure mode. A monolith avoids distributed transactions entirely.',
    ],
}

def debate_agent(position: str, round_idx: int, opponent_last: str) -> str:
    """Return a scripted argument for `position` in round `round_idx`."""
    return DEBATE_SCRIPTS[position][round_idx]

def mediator_score(argument: str) -> float:
    """Toy scoring: longer, more specific arguments score higher (proxy for 'specificity')."""
    return len(argument.split()) / 30.0   # normalise by ~30 words

def debate_coordinator(question: str, rounds: int = 3) -> str:
    print(f'Question: {question}\n')
    scores = {'pro': 0.0, 'con': 0.0}
    last = {'pro': '', 'con': ''}
    for r in range(rounds):
        print(f'--- Round {r + 1} ---')
        for position in ('pro', 'con'):
            arg = debate_agent(position, r, last['con' if position == 'pro' else 'pro'])
            score = mediator_score(arg)
            scores[position] += score
            last[position] = arg
            print(f'  [{position.upper()}] {arg}')
            print(f'        (round score: {score:.2f})')
        print()

    winner = max(scores, key=scores.get)
    synthesis = (
        f'After {rounds} rounds, both sides raised valid points. '
        f'The {winner} position scored higher ({scores[winner]:.2f} vs {scores["pro" if winner=="con" else "con"]:.2f}). '
        f'Recommendation: given team size, start with a modular monolith and extract services only '
        f'when a specific scalability bottleneck is observed.'
    )
    print(f'[Mediator synthesis]\n{synthesis}')
    return synthesis

debate_coordinator('Should the team adopt microservices for this project?')

The mediator's synthesis incorporates claims from both sides rather than just declaring a winner. This is the key quality advantage of debate: the final answer is grounded in the strongest arguments from *both* positions.

## 5. Pattern comparison

Print a summary table comparing the three cooperation patterns across key dimensions.

In [ ]:
headers = ['Pattern', 'Relative cost', 'Latency', 'Best for', 'Primary failure mode']
rows = [
    ['Voting',      'O(N) — N parallel calls', 'Low (parallel)',      'Factual QA, classification',         'Correlated errors'],
    ['Role-based',  'O(K) — K sequential steps', 'Medium (sequential)', 'Multi-step agentic tasks',           'Cascading planner error'],
    ['Debate',      'O(2K) — 2K + 1 calls',    'High (sequential rounds)', 'High-stakes ambiguous decisions', 'Consensus on a wrong answer'],
]

col_widths = [max(len(r[i]) for r in [headers] + rows) + 2 for i in range(len(headers))]

def fmt_row(row):
    return '| ' + ' | '.join(cell.ljust(col_widths[i]) for i, cell in enumerate(row)) + ' |'

sep = '|-' + '-|-'.join('-' * w for w in col_widths) + '-|'
print(fmt_row(headers))
print(sep)
for row in rows:
    print(fmt_row(row))

## ✏️ Your turn

**Challenge: implement `WeightedVoting`.**

Majority vote treats all agents equally. But agents differ — some are more accurate than others. **Weighted voting** scales each agent's vote by a confidence score, so more reliable agents have more influence on the outcome.

Given:
- `votes`: a list of `(label, confidence)` tuples, where `confidence` is a float in `[0, 1]`
- Return the label with the **highest total weight** (sum of confidences for that label)

Example: `[("yes", 0.9), ("no", 0.6), ("yes", 0.5)]` → `"yes"` (total weight 1.4 vs 0.6)

In [ ]:
def weighted_vote(votes: list[tuple[str, float]]) -> str:
    """
    Return the label with the highest total confidence-weighted vote.

    Args:
        votes: list of (label, confidence) pairs

    Returns:
        The winning label.
    """
    # TODO(you): accumulate confidence scores per label, return the label with the max total.
    pass

# --- assert cell (runs silently when correct) ---
assert weighted_vote([('yes', 0.9), ('no', 0.6), ('yes', 0.5)]) == 'yes'
assert weighted_vote([('A', 0.3), ('B', 0.8), ('A', 0.3)]) == 'B'
assert weighted_vote([('X', 1.0)]) == 'X'
assert weighted_vote([('yes', 0.5), ('no', 0.5), ('yes', 0.5)]) == 'yes'
print('passed ✓')

<details><summary>Solution</summary>

```python
def weighted_vote(votes: list[tuple[str, float]]) -> str:
    totals: dict[str, float] = {}
    for label, confidence in votes:
        totals[label] = totals.get(label, 0.0) + confidence
    return max(totals, key=totals.get)
```

</details>